# CrowdHumanでYOLO人物検出モデルをファインチューニング

CrowdHumanの **vbox（人物の見えている範囲）** を使い、`person` の1クラスの検出モデルを作成します。

このノートブックでは、まずGPU・Google Drive・保存済みファイルだけを確認します。
各セルを上から順に実行してください。**セル6の出力を確認してから、次の処理を追加します。**
保存済みtarの先頭の構成まで確認します。画像展開、注釈解析、ラベル変換、モデル取得、学習はまだ行いません。

- データ保存場所: `/content/drive/MyDrive/crowdhuman`
- 学習結果の保存先: `/content/drive/MyDrive/crowdhuman_vbox_ft/runs`
- 事前学習重みの保存先: `/content/drive/MyDrive/crowdhuman_vbox_ft/weights`
- 今後のローカルデータ展開先: `/content/crowdhuman_vbox_yolo`（現段階では作成しません）


## 1. GPUを確認

Colabの「ランタイム」→「ランタイムのタイプを変更」でGPUを選択してから実行してください。
`nvidia-smi` の結果と、PyTorchから見えるGPU・メモリ・バージョンを表示します。


In [ ]:
import subprocess
import torch

try:
    result = subprocess.run(
        ["nvidia-smi"], capture_output=True, text=True, check=False
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
except FileNotFoundError:
    print("nvidia-smiが見つかりません。ColabのGPU設定を確認してください。")

print(f"PyTorch: {torch.__version__}")
print(f"PyTorchのCUDAバージョン: {torch.version.cuda}")
print(f"CUDA利用可能: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        properties = torch.cuda.get_device_properties(index)
        print(f"GPU {index}: {properties.name}")
        print(f"GPU {index}の総メモリ: {properties.total_memory / 1024**3:.2f} GiB")
else:
    print("GPUが利用できません。ランタイムのタイプを確認してください。")


## 2. Google Driveを接続

画面の案内に従って、CrowdHumanを保存しているGoogleアカウントのDriveを接続してください。


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 3. 保存先とColabの空き容量を確認

学習結果用と事前学習重み用のフォルダを作成します。既存のファイルは削除しません。
データ保存フォルダとローカルデータ展開先は、このセルでは作成しません。
表示する空き容量は **Colabローカル** の容量です。


In [ ]:
from pathlib import Path
import shutil

DRIVE_ROOT = Path("/content/drive/MyDrive")
DATA_DIR = DRIVE_ROOT / "crowdhuman"
OUTPUT_ROOT = DRIVE_ROOT / "crowdhuman_vbox_ft"
RUNS_DIR = OUTPUT_ROOT / "runs"
WEIGHTS_DIR = OUTPUT_ROOT / "weights"
LOCAL_DATA_DIR = Path("/content/crowdhuman_vbox_yolo")

if not DRIVE_ROOT.is_dir():
    raise RuntimeError("Google Driveが接続されていません。先にセル2を実行してください。")

for directory in (RUNS_DIR, WEIGHTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"データ保存場所: {DATA_DIR}")
print(f"学習結果の保存先: {RUNS_DIR}")
print(f"事前学習重みの保存先: {WEIGHTS_DIR}")
print(f"今後のローカルデータ展開先: {LOCAL_DATA_DIR}")
usage = shutil.disk_usage("/content")
print(f"Colabローカルの総容量: {usage.total / 1024**3:.2f} GiB")
print(f"Colabローカルの空き容量: {usage.free / 1024**3:.2f} GiB")


## 4. 展開済みCrowdHumanのフォルダ構成を確認

データフォルダ直下に `checkpoints`、`logs`、`prepared`、`raw` があることを確認済みです。
画像と注釈の保存場所を調べるため、`raw` と `prepared` の中を最大20件ずつ表示します。
さらに、表示された子フォルダのうち最大5個について、中身を最大5件ずつ表示します。
表示順はファイルシステムの順序で、上限に達した場合は全件を表示していません。

このセルは一覧とファイルサイズの確認のみです。画像・注釈の内容の読み込み、データの変更は行いません。
`checkpoints` と `logs` の中身は確認しません。
確認済みの構成: 注釈は `raw/annotations`、ZIPは `raw/archives` にあり、`prepared` にはtarファイル3個と `latest.json` があります。次のセルで `latest.json` を確認します。


In [ ]:
from itertools import islice


def show_entries(directory, limit, indent="  "):
    # 大量の画像があっても、一覧取得は上限+1件で止めます。
    entries = list(islice(directory.iterdir(), limit + 1))
    shown = entries[:limit]
    for entry in shown:
        if entry.is_dir():
            print(f"{indent}[フォルダ] {entry.name}")
        elif entry.is_file():
            print(f"{indent}[ファイル] {entry.name} ({entry.stat().st_size:,} bytes)")
        else:
            print(f"{indent}[その他] {entry.name}")
    if not entries:
        print(f"{indent}（空のフォルダです）")
    if len(entries) > limit:
        print(f"{indent}（表示上限{limit}件に達しました。続きは省略しています）")
    return shown


if not DATA_DIR.is_dir():
    print(f"データフォルダが見つかりません: {DATA_DIR}")
    print("Driveの接続状態と保存場所を確認してください。")
else:
    for name in ("raw", "prepared"):
        directory = DATA_DIR / name
        print(f"\n確認先: {directory}")
        if not directory.is_dir():
            print("  フォルダが見つかりません。")
            continue
        shown = show_entries(directory, limit=20)
        children = [entry for entry in shown if entry.is_dir()]
        for child in children[:5]:
            print(f"\n  子フォルダ: {child}")
            show_entries(child, limit=5, indent="    ")
        if len(children) > 5:
            print("  （子フォルダの詳細表示は最大5個までです）")

print("\n次のセル5でprepared/latest.jsonを確認してください。")


## 5. 保存済みデータの管理情報を確認

`prepared/latest.json` の内容を表示します。どのtarファイルを参照しているか、どのような管理情報があるかを確認する段階です。
JSON内のパスを使ったファイル操作は行いません。tarの読み込み・展開や注釈解析もまだ行いません。

確認済みの管理情報では、`prepared_9c497bb835b6435ea77e0992ab70b7d4.tar` を参照し、`settings.modes` は `["vbox"]` です。次のセルで参照先tarの先頭を確認します。


In [ ]:
import json

LATEST_PATH = DATA_DIR / "prepared" / "latest.json"
print(f"確認先: {LATEST_PATH}")

if not LATEST_PATH.is_file():
    print("latest.jsonが見つかりません。保存場所を確認してください。")
else:
    # 想定外に大きいファイルを読み込まないよう、読み取り量を制限します。
    with LATEST_PATH.open("rb") as file:
        raw = file.read(64 * 1024 + 1)
    if len(raw) > 64 * 1024:
        raise ValueError("latest.jsonが64 KiBを超えています。内容の確認をいったん止めます。")
    latest_metadata = json.loads(raw.decode("utf-8-sig"))
    print(json.dumps(latest_metadata, ensure_ascii=False, indent=2))

print("\n次のセル6で参照先tarの先頭を確認してください。")


## 6. 参照先tarの先頭の構成を確認

セル5を実行してから、このセルを実行してください。
管理情報が参照するtarのファイルサイズを照合し、先頭30件までの名前・種類・サイズを表示します。
先頭の情報だけでは、学習・検証データやラベルがすべて揃っているかは判断できません。

ストリームで先頭から読むため、表示対象より前にあるファイルのデータも読み飛ばします。Driveの速度や格納順によって時間がかかる場合があります。
全体のSHA-256計算、ファイル展開、画像・ラベルの内容確認はまだ行いません。

**このセルの出力を共有し、ここで止めてください。**


In [ ]:
import tarfile
from itertools import islice

# セル5で読み込んだ管理情報から、prepared直下のtarだけを参照します。
if not isinstance(latest_metadata, dict):
    raise ValueError("latest.jsonの形式を確認してください。")
archive_name = latest_metadata.get("archive")
if (
    not isinstance(archive_name, str)
    or not archive_name
    or Path(archive_name).name != archive_name
    or "/" in archive_name
    or "\\" in archive_name
    or not archive_name.endswith(".tar")
):
    raise ValueError("archiveがprepared直下のtarファイル名ではありません。")

archive_path = DATA_DIR / "prepared" / archive_name
if not archive_path.is_file():
    raise FileNotFoundError(f"参照先tarが見つかりません: {archive_path}")

actual_size = archive_path.stat().st_size
expected_size = latest_metadata.get("size")
print(f"確認先: {archive_path}")
print(f"実ファイルのサイズ: {actual_size:,} bytes")
print(f"管理情報のサイズ: {expected_size}")
if type(expected_size) is not int or actual_size != expected_size:
    raise ValueError("管理情報と実ファイルのサイズが一致しません。ここで確認を止めます。")

print("\n先頭の最大30件（tarに格納された順）:")
member_count = 0
# 非圧縮tarをストリームで読み、先頭30件で止めます。ファイルは展開しません。
with tarfile.open(archive_path, mode="r|") as archive:
    for member in islice(archive, 30):
        member_count += 1
        if member.isdir():
            kind = "フォルダ"
        elif member.isfile():
            kind = "ファイル"
        else:
            kind = "その他"
        print(f"  [{kind}] {member.name!r} ({member.size:,} bytes)")

print(f"\n表示件数: {member_count}（上限30件。総件数は未確認です）")
print("SHA-256とtar全体の完全性、画像・ラベルの内容は未検証です。")
print("ここで出力を共有してください。データの展開はまだ行いません。")
